In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\desha\Data science\fraud_transactions\Fraud.csv")

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
df.tail()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.0,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.0,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.0,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.0,C2080388513,0.00,0.00,1,0
6362619,743,CASH_OUT,850002.52,C1280323807,850002.52,0.0,C873221189,6510099.11,7360101.63,1,0


In [5]:
df.shape

(6362620, 11)

In [6]:
print(df.isnull().sum())


step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [7]:
print(df.dtypes)

step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


In [8]:
print(df.duplicated().sum())

0



QUESTION 1: DATA CLEANING

1. Missing Values: No missing values found - dataset is already clean
2. Outliers: Amount field has extreme values but these are valid in fraud detection
3. Multi-collinearity: High correlation between old/new balances is expected (natural relationship)
    No problematic multi-collinearity among independent features


making new columns

In [9]:
df['balance_change_orig'] = df['newbalanceOrig'] - df['oldbalanceOrg']
df['balance_change_dest'] = df['newbalanceDest'] - df['oldbalanceDest']
df['transaction_hour'] = df['step'] % 24
df['is_merchant_dest'] = df['nameDest'].str.startswith('M').astype(int)

# Transaction patterns
df['zero_balance_after'] = (df['newbalanceOrig'] == 0).astype(int)
df['account_empty'] = ((df['oldbalanceOrg'] > 0) & (df['newbalanceOrig'] == 0)).astype(int)

QUESTION 3: VARIABLE SELECTION

 Variables selected based on:
 1. Domain knowledge of fraud patterns
 2. Feature importance from exploratory analysis
 3. Removal of identifiers (nameOrig, nameDest) that don't generalize
 4. Creation of engineered features capturing fraud behavior patterns


In [10]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_change_orig,balance_change_dest,transaction_hour,is_merchant_dest,zero_balance_after,account_empty
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,-9839.64,0.0,1,1,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,-1864.28,0.0,1,1,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,-181.00,0.0,1,0,1,1
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,-181.00,-21182.0,1,0,1,1
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,-11668.14,0.0,1,1,0,0


In [11]:
# Drop customer IDs (too many unique values)
df = df.drop(['nameOrig', 'nameDest'], axis=1)

In [12]:
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_change_orig,balance_change_dest,transaction_hour,is_merchant_dest,zero_balance_after,account_empty
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0,-9839.64,0.0,1,1,0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0,-1864.28,0.0,1,1,0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,0,-181.00,0.0,1,0,1,1
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,0,-181.00,-21182.0,1,0,1,1
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0,-11668.14,0.0,1,1,0,0


In [13]:
# One-hot encode transaction type
df = pd.get_dummies(df, columns=['type'], prefix='type', dtype=int)


In [14]:
df.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_change_orig,balance_change_dest,transaction_hour,is_merchant_dest,zero_balance_after,account_empty,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0,-9839.64,0.0,1,1,0,0,0,0,0,1,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0,-1864.28,0.0,1,1,0,0,0,0,0,1,0
2,1,181.00,181.0,0.00,0.0,0.0,1,0,-181.00,0.0,1,0,1,1,0,0,0,0,1
3,1,181.00,181.0,0.00,21182.0,0.0,1,0,-181.00,-21182.0,1,0,1,1,0,1,0,0,0
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0,-11668.14,0.0,1,1,0,0,0,0,0,1,0


In [15]:
df.isFraud.value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [16]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop('isFraud', axis=1)
y = df['isFraud']

# Apply SMOTE only on training data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [17]:
y_train_res.value_counts()

isFraud
0    4448056
1    4448056
Name: count, dtype: int64

In [18]:
X_train_res.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,balance_change_orig,balance_change_dest,transaction_hour,is_merchant_dest,zero_balance_after,account_empty,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,202,26771.98,27521.00,54292.98,7175503.03,7148731.05,0,26771.98,-26771.98,10,0,0,0,1,0,0,0,0
1,36,344879.65,0.00,0.00,3731846.57,4076726.22,0,0.00,344879.65,12,0,1,0,0,1,0,0,0
2,20,1862607.84,0.00,0.00,3329828.01,4913631.27,0,0.00,1583803.26,20,0,1,0,0,0,0,0,1
3,354,37739.35,66684.93,28945.58,0.00,0.00,0,-37739.35,0.00,18,1,0,0,0,0,0,1,0
4,38,376055.13,22501.00,0.00,182350.43,558405.55,0,-22501.00,376055.12,14,0,1,1,0,1,0,0,0


In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

In [20]:
X_train_scaled

array([[-0.60588034, -0.42784771, -0.37804125, ..., -0.05718147,
        -0.45138222, -0.63939537],
       [-1.56522296, -0.259703  , -0.38653779, ..., -0.05718147,
        -0.45138222, -0.63939537],
       [-1.65768972,  0.5425347 , -0.38653779, ..., -0.05718147,
        -0.45138222,  1.56397755],
       ...,
       [-0.25335082, -0.35775653, -0.33733393, ..., -0.05718147,
        -0.45138222, -0.63939537],
       [ 0.46904573,  3.49628414,  1.91371776, ..., -0.05718147,
        -0.45138222, -0.63939537],
       [ 0.7695627 ,  2.69306593,  1.44457748, ..., -0.05718147,
        -0.45138222, -0.63939537]])

In [21]:
X_train_scaled.shape

(8896112, 18)

In [22]:
X_train_res.shape

(8896112, 18)

QUESTION 2: FRAUD DETECTION MODEL DESCRIPTION
1. Model: Ensemble-based XGBoost classifier
2. Approach: Supervised learning with balanced classes using SMOTE
3. Features: 18 engineered features capturing transaction patterns
4. Algorithm: Gradient boosting optimized for imbalanced data
5. Output: Probability score (0-1) indicating fraud likelihood

In [23]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

models = {
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=12,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=10,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',  # Faster for large data
        eval_metric='logloss'
    )
    
}

In [24]:
for model_name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {model_name}...")
    print(f"{'='*50}")
    
    
    # Train the model
    model.fit(X_train_scaled, y_train_res)


Training LightGBM...

Training XGBoost...


QUESTION 4: MODEL PERFORMANCE DEMONSTRATION

1. Using best tools: ROC-AUC, Precision-Recall, Classification Report
2. XGBoost shows superior performance with 99.91% accuracy and 99.91% ROC-AUC
3. Key metric: Fraud F1-score of 0.73 (balanced precision and recall)

In [26]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
# Evaluate LightGBM
print("LightGBM Evaluation:")
print("=" * 50)

y_pred_lgb = models['LightGBM'].predict(X_test_scaled)
y_pred_proba_lgb = models['LightGBM'].predict_proba(X_test_scaled)[:, 1]

accuracy_lgb = accuracy_score(y_test, y_pred_lgb)
roc_auc_lgb = roc_auc_score(y_test, y_pred_proba_lgb)

print(f"Accuracy: {accuracy_lgb:.4f}")
print(f"ROC-AUC: {roc_auc_lgb:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lgb))

# Evaluate XGBoost
print("\n" + "=" * 50)
print("XGBoost Evaluation:")
print("=" * 50)

y_pred_xgb = models['XGBoost'].predict(X_test_scaled)
y_pred_proba_xgb = models['XGBoost'].predict_proba(X_test_scaled)[:, 1]

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
roc_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)

print(f"Accuracy: {accuracy_xgb:.4f}")
print(f"ROC-AUC: {roc_auc_xgb:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

# Quick comparison
print("\n" + "=" * 50)
print("QUICK COMPARISON:")
print("=" * 50)
print(f"LightGBM - Accuracy: {accuracy_lgb:.4f}, ROC-AUC: {roc_auc_lgb:.4f}")
print(f"XGBoost  - Accuracy: {accuracy_xgb:.4f}, ROC-AUC: {roc_auc_xgb:.4f}")

if accuracy_lgb > accuracy_xgb:
    print("LightGBM performs better on Accuracy")
elif accuracy_xgb > accuracy_lgb:
    print("XGBoost performs better on Accuracy")
else:
    print("Both models have same Accuracy")

if roc_auc_lgb > roc_auc_xgb:
    print("LightGBM performs better on ROC-AUC")
elif roc_auc_xgb > roc_auc_lgb:
    print("XGBoost performs better on ROC-AUC")
else:
    print("Both models have same ROC-AUC")

LightGBM Evaluation:
Accuracy: 0.9959
ROC-AUC: 0.9990

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906351
           1       0.23      0.99      0.38      2435

    accuracy                           1.00   1908786
   macro avg       0.62      0.99      0.69   1908786
weighted avg       1.00      1.00      1.00   1908786


XGBoost Evaluation:
Accuracy: 0.9991
ROC-AUC: 0.9991

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906351
           1       0.58      0.98      0.73      2435

    accuracy                           1.00   1908786
   macro avg       0.79      0.99      0.87   1908786
weighted avg       1.00      1.00      1.00   1908786


QUICK COMPARISON:
LightGBM - Accuracy: 0.9959, ROC-AUC: 0.9990
XGBoost  - Accuracy: 0.9991, ROC-AUC: 0.9991
XGBoost performs better on Accuracy
XGBoost performs better on ROC-AUC


QUESTION 5: KEY FRAUD PREDICTION FACTORS

  Top factors from feature importance analysis:
 1. amount: Transaction value
 2. account_empty: Account emptying pattern
 3. balance_change_orig: Origin balance change
 4. oldbalanceOrg: Initial origin balance
 5. type_TRANSFER: Transaction type
 6. zero_balance_after: Zero balance after transaction

  QUESTION 6: DO THESE FACTORS MAKE SENSE?

 #YES, they make perfect sense:
 1. Large amounts are common in fraud to maximize gain
 2. Account emptying is classic fraud behavior
 3. TRANSFER type is primary method for moving stolen funds
 4. Balance changes indicate fund movement patterns
 5. Zero balance after transaction confirms complete theft
 
  All factors align with known fraud patterns in financial transactions

In [27]:
def predict_multiple_transactions(model, scaler, transactions_df):
    """
    Predict fraud for multiple transactions in a DataFrame
    transactions_df: DataFrame with same columns as training features
    """
    # Scale the features
    transactions_scaled = scaler.transform(transactions_df)
    
    # Make predictions
    predictions = model.predict(transactions_scaled)
    probabilities = model.predict_proba(transactions_scaled)[:, 1]
    
    # Create results DataFrame
    results_df = transactions_df.copy()
    results_df['fraud_prediction'] = predictions
    results_df['fraud_probability'] = probabilities
    
    return results_df

# Example: Create sample new transactions
import pandas as pd

new_transactions = pd.DataFrame([
    [100, 5000.00, 10000.00, 5000.00, 0.00, 5000.00, 0, -5000.00, 5000.00, 14, 0, 0, 0, 0, 1, 0, 0, 0],
    [200, 100.00, 500.00, 400.00, 1000.00, 1100.00, 0, -100.00, 100.00, 10, 0, 0, 0, 0, 0, 0, 1, 0],
    [50, 10000.00, 10000.00, 0.00, 0.00, 10000.00, 0, -10000.00, 10000.00, 3, 0, 1, 1, 0, 0, 0, 0, 1]
], columns=X_train_res.columns)  # Use the same column names as your training data

# Make predictions
predictions_df = predict_multiple_transactions(models['XGBoost'], scaler, new_transactions)
print("Predictions for new transactions:")
print(predictions_df[['fraud_prediction', 'fraud_probability']])

Predictions for new transactions:
   fraud_prediction  fraud_probability
0                 0           0.000118
1                 0           0.001434
2                 1           0.991329


 QUESTION 7: PREVENTION MEASURES FOR INFRASTRUCTURE UPDATE

 1. Real-time scoring: Integrate model API for instant fraud scoring
 2. Threshold tuning: Adjust based on business risk tolerance
 3. Multi-layered defense: Combine with rule-based systems
 4. Continuous monitoring: Retrain model with new fraud patterns
 5. Feature store: Maintain consistent feature engineering pipeline
 6. Alert system: Automatic flags for high-risk transactions



 QUESTION 8: MEASURING IMPLEMENTATION EFFECTIVENESS

 Success metrics to track:
 1. Fraud detection rate: % of actual fraud caught
 2. False positive rate: % of legitimate transactions flagged
 3. Average fraud loss: Reduction in financial losses
 4. Response time: Speed of fraud detection and prevention
 5. Model drift: Monitor performance degradation over time
 6. Business impact: Customer satisfaction and trust metrics

In [29]:
import joblib

# Save the trained model
joblib.dump(models['XGBoost'], 'fraud_detection_xgboost_model.pkl')
print("XGBoost model saved successfully!")

XGBoost model saved successfully!
